## Uniform Int4@4.0 — Tweak Run (3,000 Steps)

**Goal:** Achieve < 16.0 MB via MLP_MULT 4.1 and optimized pruning.

| Cell | Purpose | Est. Time |
|------|---------|----------|
| 1 | Setup: clone, deps, data | ~3 min |
| 2 | Patch: paths + T4 SDPA + DDP compat | ~5 sec |
| 3 | Train: 3k steps on 2x T4 (MLP_MULT: 4.1) | ~4.5 hrs |

In [ ]:
# Cell 1/3 — Setup
import os, subprocess, shutil
from pathlib import Path

os.chdir('/kaggle/working')
for d in ['pg', 'openai-pg']:
    if os.path.exists(d): shutil.rmtree(d)

print('=== Cloning repositories ===')
!git clone --depth 1 https://github.com/jmoncayo-pursuit/parameter-golf-uniform-int4.git pg
!git clone --depth 1 https://github.com/openai/parameter-golf.git openai-pg

print('\n=== Installing dependencies ===')
%pip install -q sentencepiece zstandard huggingface_hub

print('\n=== Downloading data shard ===')
os.chdir('/kaggle/working/openai-pg')
!python3 data/cached_challenge_fineweb.py --variant sp1024 --train-shards 1

tok = Path('/kaggle/working/openai-pg/data/tokenizers/fineweb_1024_bpe.model')
data = Path('/kaggle/working/openai-pg/data/datasets/fineweb10B_sp1024')
script = Path('/kaggle/working/pg/train_gpt.py')
assert tok.exists(), f'Missing: {tok}'
assert data.exists(), f'Missing: {data}'
assert script.exists(), f'Missing: {script}'
print(f'\n✅ SETUP COMPLETE')

In [ ]:
# Cell 2/3 — Patch train_gpt.py for Kaggle T4 x2
from pathlib import Path
import torch

train_script = Path('/kaggle/working/pg/train_gpt.py')
content = train_script.read_text()
patches_applied = []

# --- Patch 1: Data paths ---
for old, new in [
    ('./data/tokenizers/fineweb_1024_bpe.model',
     '/kaggle/working/openai-pg/data/tokenizers/fineweb_1024_bpe.model'),
    ('./data/datasets/fineweb10B_sp1024',
     '/kaggle/working/openai-pg/data/datasets/fineweb10B_sp1024'),
]:
    if old in content:
        content = content.replace(old, new)
        patches_applied.append(f'path: {old}')

# --- Patch 2: SDPA backend for T4 (SM 7.5 — no flash attention) ---
old_sdpa = (
    '    if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8:\n'
    '        enable_cudnn_sdp(False)\n'
    '        enable_flash_sdp(True)\n'
    '        enable_mem_efficient_sdp(False)\n'
    '        enable_math_sdp(False)\n'
)
new_sdpa = (
    '    if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8:\n'
    '        enable_cudnn_sdp(False)\n'
    '        enable_flash_sdp(True)\n'
    '        enable_mem_efficient_sdp(False)\n'
    '        enable_math_sdp(False)\n'
    '    elif torch.cuda.is_available():\n'
    '        enable_cudnn_sdp(False)\n'
    '        enable_flash_sdp(False)\n'
    '        enable_mem_efficient_sdp(True)\n'
    '        enable_math_sdp(True)\n'
)
if old_sdpa in content:
    content = content.replace(old_sdpa, new_sdpa, 1)
    patches_applied.append('SDPA T4 fallback')
elif 'elif torch.cuda.is_available():' in content:
    patches_applied.append('SDPA T4 fallback (already present)')

# --- Patch 3: DDP init_process_group compat ---
old_init = 'dist.init_process_group(backend="nccl", device_id=device)'
new_init = """try:
            dist.init_process_group(backend="nccl", device_id=device)
        except TypeError:
            dist.init_process_group(backend="nccl")"""
if old_init in content:
    content = content.replace(old_init, new_init, 1)
    patches_applied.append('DDP init_process_group compat')

train_script.write_text(content)

# Verify
verify = train_script.read_text()
assert '/kaggle/working/openai-pg/data/tokenizers' in verify, 'Path patch FAILED'

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        name = torch.cuda.get_device_name(i)
        mem = torch.cuda.get_device_properties(i).total_memory / 1024**3
        cap = torch.cuda.get_device_capability(i)
        print(f'  GPU {i}: {name} ({mem:.1f} GB, SM {cap[0]}.{cap[1]})')

print(f'\nPatches applied ({len(patches_applied)}):')
for p in patches_applied:
    print(f'  ✅ {p}')
print('\n✅ PATCH COMPLETE')

In [ ]:
# Cell 3/3 — Train: 3,000-step (Tweak Run)
import os, subprocess, time, gc, torch

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

n_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
print(f'GPUs available: {n_gpus}')

os.chdir('/kaggle/working/pg')

env = os.environ.copy()
env.update({
    # Training — T4-safe batch size
    'ITERATIONS':         '3000',
    'TRAIN_SEQ_LEN':      '1024',
    'TRAIN_BATCH_TOKENS': '65536',
    'WARMUP_STEPS':       '20',
    'WARMDOWN_ITERS':     '600',
    'SEED':               '42',
    # --- Tweak Run Specs (User Recommended) ---
    'MLP_MULT':           '4.1',
    'COMPRESSOR':         'lzma',
    # -----------------------
    # Validation
    'VAL_LOSS_EVERY':     '500',
    'VAL_MAX_TOKENS':     '65536',
    'VAL_BATCH_SIZE':     '65536',
    # Logging
    'TRAIN_LOG_EVERY':    '50',
    # Safety
    'MAX_WALLCLOCK_SECONDS': '39600',
    # Eval
    'EVAL_STRIDE':        '64',
    'EVAL_BATCH_SEQS':    '8', 
    # Env
    'PYTHONUNBUFFERED':   '1',
    'NCCL_P2P_DISABLE':   '1',
    'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True',
})

if n_gpus >= 2:
    cmd = ['python3', '-m', 'torch.distributed.run',
           '--nproc_per_node=2', '--master_port=29500',
           'train_gpt.py']
    mode = 'DDP (2x T4)'
else:
    cmd = ['python3', '-u', 'train_gpt.py']
    mode = f'Single GPU ({n_gpus})'

print(f'Mode: {mode}')
print(f'ITERATIONS={env["ITERATIONS"]}  BATCH={env["TRAIN_BATCH_TOKENS"]}  SEQ={env["TRAIN_SEQ_LEN"]}')
print(f'MLP_MULT={env["MLP_MULT"]}  COMPRESSOR={env["COMPRESSOR"]}')
print('=' * 60)

t0 = time.time()
proc = subprocess.run(cmd, env=env)
elapsed = time.time() - t0

print(f'\n{"=" * 60}')
print(f'Finished in {elapsed/3600:.2f} hrs (exit code: {proc.returncode})')

from pathlib import Path
for f in ['final_model.pt', 'final_model.mixed.ptz', 'final_summary.json', 'final_summary.md']:
    p = Path(f'/kaggle/working/pg/{f}')
    if p.exists():
        print(f'  ✅ {f}: {p.stat().st_size / 1024 / 1024:.2f} MB')
    else:
        print(f'  ❌ {f}: NOT FOUND')

summary = Path('/kaggle/working/pg/final_summary.md')
if summary.exists():
    print(f'\n--- Final Summary ---')
    print(summary.read_text())